In [1]:
import torch
from torch.utils.data import DataLoader

In [2]:
# need to add path using os and sys first
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

In [3]:
from utils.tokenizer import prepare_tokenizer, pad_tensor
from utils.load_data import load_data, prep_dolly_collate_fn, prep_squad_collate_fn

In [4]:
vocab, eos_idx, bos_idx, vocab_size = prepare_tokenizer()

In [5]:
qc_len = 512 + 256
a_len = 100

In [6]:
squad_collate = prep_squad_collate_fn(qc_len, a_len, eos_idx, bos_idx, pad_tensor)

In [7]:
squad, dolly = load_data()

In [8]:
batch_size = 2

In [9]:
squadDataloader = DataLoader(squad, batch_size=batch_size, shuffle=True, collate_fn=squad_collate)

In [10]:
embedding_dim = 128
num_heads = 4

In [11]:
from models.embedding import StableEmbedding
from models.phm import phm

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [13]:
for batch in squadDataloader:
    break

In [14]:
emb = StableEmbedding(vocab_size, embedding_dim)

In [15]:
emb = emb.to(device)

In [16]:
qc, a = batch

In [17]:
a_input = a[:, :-1]
a_target = a[:, 1:]

In [18]:
qc = qc.to(device)
a_input = a_input.to(device)
a_target = a_target.to(device)

In [19]:
# encoder layer starts here
qc_emb = emb(qc)

In [20]:
qc_proj_q = phm(4, embedding_dim, embedding_dim)
qc_proj_q = qc_proj_q.to(device)
qc_proj_kv = phm(4, embedding_dim, embedding_dim)
qc_proj_kv = qc_proj_kv.to(device)

In [21]:
qc_q_heads = qc_proj_q(qc_emb).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)

In [22]:
qc_kv_heads = qc_proj_kv(qc_emb).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)

In [23]:
attn_mask = (qc != eos_idx).float()

In [24]:
attn_mask

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.]], device='cuda:0')

In [25]:
attn_mask.logical_not()

tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]], device='cuda:0')

In [26]:
attn_mask.masked_fill_(attn_mask.logical_not(), float('-inf'))

tensor([[1., 1., 1.,  ..., -inf, -inf, -inf],
        [1., 1., 1.,  ..., -inf, -inf, -inf]], device='cuda:0')

In [27]:
attn_mask.masked_fill_(attn_mask == 1, 0)

tensor([[0., 0., 0.,  ..., -inf, -inf, -inf],
        [0., 0., 0.,  ..., -inf, -inf, -inf]], device='cuda:0')

In [28]:
# attn_mask right now is batch_size x qc_len, but we need batch_size x num_heads x qc_len x qc_len
attn_mask = attn_mask.unsqueeze(1).unsqueeze(2).repeat(1, num_heads, qc_len, 1)

In [29]:
attn_mask.shape

torch.Size([2, 4, 768, 768])

In [30]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    out = torch.nn.functional.scaled_dot_product_attention(qc_q_heads, qc_kv_heads, qc_kv_heads, attn_mask)

In [31]:
out.shape

torch.Size([2, 4, 768, 32])

In [32]:
out_proj = phm(4, embedding_dim, embedding_dim)
out_proj = out_proj.to(device)

In [33]:
projected = out_proj(out.transpose(1,2).reshape(batch_size, -1, embedding_dim))

In [34]:
projected.shape == qc_emb.shape

True

In [35]:
# start decoder layer
a_input_emb = emb(a_input)

In [36]:
a_input_proj_q = phm(4, embedding_dim, embedding_dim)
a_input_proj_q = a_input_proj_q.to(device)
a_input_proj_kv = phm(4, embedding_dim, embedding_dim)
a_input_proj_kv = a_input_proj_kv.to(device)

In [37]:
a_input_q_heads = a_input_proj_q(a_input_emb).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)
a_input_kv_heads = a_input_proj_kv(a_input_emb).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)

In [38]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    a_out = torch.nn.functional.scaled_dot_product_attention(a_input_q_heads, a_input_kv_heads, a_input_kv_heads, is_causal=True)

In [39]:
a_out.shape

torch.Size([2, 4, 99, 32])

In [40]:
a_out_proj = phm(4, embedding_dim, embedding_dim)
a_out_proj = a_out_proj.to(device)

In [41]:
projected_a_out = a_out_proj(a_out.transpose(1,2).reshape(batch_size, -1, embedding_dim))

In [42]:
projected_a_out.shape == a_input_emb.shape

True

In [43]:
# now perform cross attention using projected_a_out as query and projected as key and value
cross_proj_q = phm(4, embedding_dim, embedding_dim)
cross_proj_q = cross_proj_q.to(device)
cross_proj_kv = phm(4, embedding_dim, embedding_dim)
cross_proj_kv = cross_proj_kv.to(device)

In [44]:
a_input_q_cross_heads = cross_proj_q(projected_a_out).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)
projected_kv_cross_heads = cross_proj_kv(projected).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)

In [45]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    a_out_cross = torch.nn.functional.scaled_dot_product_attention(a_input_q_cross_heads, projected_kv_cross_heads, projected_kv_cross_heads)
    # note that this is not causal because the decoder can see the entire encoder output

In [49]:
# experiment, take the embedding of the first token of decoder
first_token = projected_a_out[:, 0, :].unsqueeze(1)

In [51]:
# what if we use this as the kv for cross attention while using the entire projected as q
# so we have cross attention from the decoder to the encoder instead of the other way around
first_token_proj_kv = phm(4, embedding_dim, embedding_dim)
first_token_proj_kv = first_token_proj_kv.to(device)
projected_q = phm(4, embedding_dim, embedding_dim)
projected_q = projected_q.to(device)

In [52]:
first_token_kv_heads = first_token_proj_kv(first_token).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)
projected_q_heads = projected_q(projected).view(batch_size, -1, num_heads, embedding_dim // num_heads).transpose(1, 2)

In [53]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    first_token_cross = torch.nn.functional.scaled_dot_product_attention(projected_q_heads, first_token_kv_heads, first_token_kv_heads)
    # note that this isn't causal either, because while the encoder might not be able to
    # see every token in the decoder, it is guaranteed to see the first token,
    # so there is no information leaking